# 03. 시나리오와 민감도

시나리오 = `{요인 키: 기준 대비 이동}`. 만드는 규칙은 보통 함수다. 여러 객체의 요인을 한 시나리오에 함께 담을 수 있어서, 변수 사이 연동(PCA 로딩, 나중에 spot·vol SSR 규칙, vol 피팅 파라미터 이동)도 같은 틀로 넣는다.

| 단계 | 코드 |
|---|---|
| 시나리오 규칙 | `src/quanttool/analysis/scenarios.py` (`parallel`, `bucket`, `from_loadings`, `combine`) |
| 적용·해제 | `Market.apply`, `reset`, `with market.scenario(s):` |
| 민감도·손익 | `src/quanttool/analysis/risk.py` (`key_rate`, `cross_gamma`, `scenario_pnl`), `configs/risk.yaml` |

> 데이터: `tests/fixtures/market_sample.txt`는 **합성 샘플**(`날짜 키 값`)이다. 흐름을 익히는 용도이고 숫자로 시장을 해석하지 않는다.

In [ ]:
from datetime import date
from pathlib import Path

import pandas as pd

from quanttool.config import load_config
from quanttool.market import build_market
from quanttool.market.quotes import read_quotes

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
cfg = load_config(ROOT / "configs")
quotes = read_quotes(ROOT / "tests" / "fixtures" / "market_sample.txt")  # 합성 샘플
ASOF = date(2026, 10, 23)

from quanttool.analysis.risk import cross_gamma, key_rate, scenario_pnl
from quanttool.analysis.scenarios import combine, from_loadings, parallel
from quanttool.portfolio import Portfolio

market = build_market(ASOF, quotes[ASOF], cfg)
book = Portfolio.from_file(ROOT / "positions" / "ust_book.yaml", products=cfg.products).build(
    market, cfg
)

## 0. 흐름: 시나리오 생성 → 시장 적용 → 재평가 → risk

```
analysis/scenarios.py   {요인: 이동} 사전 하나를 만든다 (시장을 건드리지 않음)
        ↓
market.apply / scenario  요인 Quote를 기준 + 이동으로, 바뀐 객체와 그에 기대는 객체 refresh
        ↓
book.npvs()              연결된 상품이 Observer로 다시 가격
        ↓
analysis/risk.py         이것을 여러 번 반복: 위·아래, 만기마다, 상품별 크기·방식, 1bp 환산, 표
```

시나리오 **하나**는 risk 없이 세 줄이면 된다(아래 1). risk는 "어떻게 흔들어 몇으로 부를지"(그릭 정의)를 담는 곳이다. 그릭 수정 실험은 여기서 한다.

## 1. 손으로 만든 시나리오: risk 없이 세 줄

In [ ]:
base = book.npvs()
with market.scenario(parallel(market, "UST", 0.0010)):
    by_hand = book.npvs() - base
print("해제 후 원상태:", (book.npvs() == base).all())
pd.DataFrame(
    {
        "손으로": by_hand,
        "scenario_pnl": scenario_pnl(book, market, {"p": parallel(market, "UST", 0.0010)})
        .loc["p"]
        .drop("total"),
    }
).round(2)

### 요인 로딩 시나리오 (PCA 모양)
PCA 시나리오는 "요인 크기 × 격자점별 로딩"이다. 아래 로딩은 **설명용으로 만든 값**(단기 −1 → 장기 +1)이고 추정값이 아니다. 실제 로딩은 `configs/estimated/`에 출처와 함께 둔다.

In [ ]:
obj = market.objects["UST"]
loading = {t: -1 + 2 * time / max(obj.times.values()) for t, time in obj.times.items()}
scenarios = {
    "parallel +10bp": parallel(market, "UST", 0.0010),
    "steepener 5bp": from_loadings("UST", loading, 0.0005),
    "both": combine(parallel(market, "UST", 0.0010), from_loadings("UST", loading, 0.0005)),
}
scenario_pnl(book, market, scenarios).round(2)

## 2. key rate delta·gamma
이동 만기마다 그 점을 1bp(삼각형 가중) 올리고 내려 NPV 차이를 본다. 값은 1bp당 손익.

In [ ]:
kr = key_rate(book, market, cfg.risk, "UST")
print(
    "Σ delta",
    round(kr["delta"]["total"].sum(), 2),
    " / 평행 +1bp",
    round(scenario_pnl(book, market, {"p": parallel(market, "UST", 1e-4)}).loc["p", "total"], 2),
    "(차이 = 2차 항)",
)
kr["delta"].round(1)

In [ ]:
kr["gamma"].round(4).tail()

## 3. 상품별 민감도 규칙
`risk.yaml`의 `overrides`에 거래 id나 상품 종류별로 이동 크기·방식을 정한다. 아래는 채권 전체를 10bp 중심 차분으로 계산하고 1bp로 다시 환산한 것. 그릭 계산 규칙을 상품별로 바꾸는 자리다.

In [ ]:
from quanttool.config import Override

risk_fd = cfg.risk.model_copy(
    update={"overrides": {"Bond": Override(shift_size=0.001, scheme="Central")}}
)
fd = key_rate(book, market, risk_fd, "UST")["delta"]
pd.DataFrame({"1bp Forward": kr["delta"]["UST_10Y"], "10bp Central → 1bp": fd["UST_10Y"]}).loc[
    ["5Y", "10Y"]
].round(2)

## 4. 이동 만기를 성글게 / cross gamma
cross gamma는 두 요인을 함께 움직일 때만 생기는 손익이다. 바로 옆 격자점(9Y·10Y)은 같은 현금흐름에 함께 걸려 0이 아니고, 떨어진 점은 0이다.

In [ ]:
coarse = cfg.risk.model_copy(
    update={
        "shift_tenors": {"UST": ["2Y", "5Y", "10Y", "30Y"]},
        "cross_gamma": [("UST/9Y", "UST/10Y"), ("UST/5Y", "UST/10Y")],
    }
)
display(key_rate(book, market, coarse, "UST")["delta"].round(1))
cross_gamma(book, market, coarse)  # 겹치는 현금흐름이 없는 두 격자점(5Y·10Y)은 정확히 0